# Silver: ERP product categories
**Source:** `bronze.erp_px_cat_g1v2`  →  **Target:** `silver.erp_product_category`

**What this notebook does:**
- Remove extra spaces
- Maintenance flag: Yes/No → True/False
- Rename columns

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType

CATALOG = "workspace"

## Read the Bronze table

In [0]:
df = spark.table(f"{CATALOG}.bronze.erp_px_cat_g1v2")

## 1. Trim spaces

In [0]:
# Remove extra spaces from every text column
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, F.trim(F.col(field.name)))

## 2. Maintenance flag to True/False

In [0]:
df = df.withColumn(
    "maintenance",
    F.when(F.upper(F.col("maintenance")) == "YES", F.lit(True))
     .when(F.upper(F.col("maintenance")) == "NO",  F.lit(False))
     .otherwise(None)
)

## 3. Rename columns

In [0]:
RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag",
}
for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

## Write the Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable(f"{CATALOG}.silver.erp_product_category")

## Check it Quickly

In [0]:
result = spark.table(f"{CATALOG}.silver.erp_product_category")
print("rows:", result.count())
result.limit(10).display()